# Neagari: A Kilobyte-Scale Patch of a 1-Bit Language Model

This notebook demonstrates a post-deployment adaptation of PrismML's Bonsai 1.7B, a 1-bit quantized Qwen3-1.7B, using targeted XOR weight patches. A single 13 KB patch flips 3,019 of the model's 1.7 billion sign bits (0.00018% of all weights), correcting two specific failure modes in verbatim text extraction without degrading general capability.
This runs on a free Colab T4 GPU, the model is only 240 MB.

In [ ]:
!pip install -q gguf transformers torch numpy huggingface_hub 2>/dev/null
import os, math, json, time, zlib, base64
import torch, numpy as np
import torch.nn.functional as F

## Load the Model

In [ ]:
GROUP_SIZE = 128

def patch_gguf_q1_0():
    import gguf
    from gguf.constants import GGMLQuantizationType, GGML_QUANT_SIZES
    try: GGMLQuantizationType(41)
    except ValueError:
        obj = int.__new__(GGMLQuantizationType, 41)
        obj._name_ = 'Q1_0'; obj._value_ = 41
        GGMLQuantizationType._member_map_['Q1_0'] = obj
        GGMLQuantizationType._value2member_map_[41] = obj
    if GGMLQuantizationType(41) not in GGML_QUANT_SIZES:
        GGML_QUANT_SIZES[GGMLQuantizationType(41)] = (128, 18)

def load_model(path, device):
    patch_gguf_q1_0()
    from gguf import GGUFReader
    reader = GGUFReader(path)
    meta = {}
    for f in reader.fields:
        try:
            parts = reader.fields[f].parts
            v = parts[-1].tolist() if hasattr(parts[-1],'tolist') else parts[-1]
            if isinstance(v,list) and len(v)==1: v=v[0]
            meta[f] = v
        except: pass
    tmap = {t.name:t for t in reader.tensors}
    def _m(*keys, default):
        for k in keys:
            if k in meta: return meta[k]
        return default
    cfg = {
        'n_layers':_m('llama.block_count','qwen2.block_count','qwen3.block_count',default=28),
        'n_heads':_m('llama.attention.head_count','qwen2.attention.head_count','qwen3.attention.head_count',default=16),
        'n_kv_heads':_m('llama.attention.head_count_kv','qwen2.attention.head_count_kv','qwen3.attention.head_count_kv',default=8),
        'hidden_dim':_m('llama.embedding_length','qwen2.embedding_length','qwen3.embedding_length',default=2048),
        'rms_eps':1e-6,
        'rope_theta':_m('llama.rope.freq_base','qwen2.rope.freq_base','qwen3.rope.freq_base',default=1000000.0),
    }
    hd=_m('llama.attention.key_length','qwen2.attention.key_length','qwen3.attention.key_length',default=None)
    cfg['head_dim']=hd if hd else (cfg['hidden_dim']//cfg['n_heads'])
    gate=[n for n in tmap if 'ffn_gate' in n and 'weight' in n][0]
    gs=tuple(int(s) for s in tmap[gate].shape)
    cfg['intermediate_dim']=gs[1] if gs[0]==cfg['hidden_dim'] else gs[0]
    weights={}
    for name,tensor in tmap.items():
        weights[name]={'raw':torch.tensor(np.array(tensor.data)),
                       'shape':tuple(int(s) for s in tensor.shape),
                       'type':str(tensor.tensor_type).rsplit('.',1)[-1]}
    return cfg,weights

In [ ]:
class Q1Engine:
    EMBED='token_embd.weight'; NORM='output_norm.weight'; LM_HEAD='output.weight'
    ATTN_Q='attn_q.weight'; ATTN_K='attn_k.weight'; ATTN_V='attn_v.weight'; ATTN_O='attn_output.weight'
    FFN_GATE='ffn_gate.weight'; FFN_UP='ffn_up.weight'; FFN_DOWN='ffn_down.weight'
    ATTN_NORM='attn_norm.weight'; FFN_NORM='ffn_norm.weight'

    def __init__(self, cfg, weights, device):
        self.cfg, self.weights, self.device = cfg, weights, device
        self._shifts = torch.arange(32, device=device, dtype=torch.int32)
        self._dequant_cache, self._unpacked_cache = {}, {}
        self.tokenizer = None
        vram = torch.cuda.get_device_properties(device).total_memory/1e9 if device.type=='cuda' else 0
        nL,hD,iD = cfg['n_layers'],cfg['hidden_dim'],cfg['intermediate_dim']
        nH,nKV,hdim = cfg['n_heads'],cfg['n_kv_heads'],cfg['head_dim']
        est_gb = (151936*hD + nL*(hD*nH*hdim + hD*nKV*hdim*2 + nH*hdim*hD + hD*iD*3))*2/1e9
        self.cache_dequant = (vram - est_gb) > 3.0

    @staticmethod
    def ln(layer, suffix): return f'blk.{layer}.{suffix}'

    def unpack_q1_0(self, raw, out_f, in_f):
        rb=raw.to(torch.uint8).numpy().ravel(); ng=(int(out_f)*int(in_f))//GROUP_SIZE; bpg=GROUP_SIZE//8+2
        off=np.arange(ng)*bpg; sr=np.zeros((ng,2),dtype=np.uint8); sr[:,0]=rb[off]; sr[:,1]=rb[off+1]
        scales=sr.view(np.float16).reshape(ng); br=np.zeros((ng,16),dtype=np.uint8)
        for i in range(16): br[:,i]=rb[off+2+i]
        packed=br.view(np.uint32).reshape(ng,4).view(np.int32)
        return torch.tensor(packed,dtype=torch.int32,device=self.device),torch.tensor(scales,dtype=torch.float16,device=self.device)

    def dequantize(self, packed, scales, out_f, in_f):
        ng=packed.shape[0]
        if int(out_f)*int(in_f)<60_000_000:
            bits=((packed.unsqueeze(-1)>>self._shifts)&1).float().reshape(ng,GROUP_SIZE)
            return (scales.float().unsqueeze(1)*(2.*bits-1.)).reshape(int(out_f),int(in_f))
        chunks=[]
        for s in range(0,ng,50000):
            e=min(s+50000,ng); b=((packed[s:e].unsqueeze(-1)>>self._shifts)&1).float().reshape(e-s,GROUP_SIZE)
            chunks.append((scales[s:e].float().unsqueeze(1)*(2.*b-1.)).reshape(-1)); del b
        return torch.cat(chunks).reshape(int(out_f),int(in_f))

    def load_fp(self, name):
        w=self.weights[name]; raw=w['raw'].numpy().view(np.uint8).tobytes()
        dt=np.float16 if w['type'] in ('1','F16','f16') else np.float32
        return torch.tensor(np.frombuffer(raw,dtype=dt).reshape(w['shape']),dtype=torch.float32,device=self.device)

    def get_weight(self, name):
        if name in self._dequant_cache: return self._dequant_cache[name]
        if name not in self.weights: return None
        w=self.weights[name]
        if w['type'] in ('Q1_0','q1_0','41'):
            of=w['shape'][1] if len(w['shape'])>1 else 1; inf=w['shape'][0]
            if name not in self._unpacked_cache:
                p,s=self.unpack_q1_0(w['raw'],of,inf); self._unpacked_cache[name]=(p,s,of,inf)
            p,s,of,inf=self._unpacked_cache[name]; r=self.dequantize(p,s,of,inf)
            if self.cache_dequant: self._dequant_cache[name]=r
            return r
        r=self.load_fp(name); self._dequant_cache[name]=r; return r

    def linear(self, x, name):
        w=self.get_weight(name); r=x@w.t()
        if not self.cache_dequant: del w; torch.cuda.empty_cache()
        return r

    @staticmethod
    def rms_norm(x,w,eps=1e-6): return x*torch.rsqrt(x.pow(2).mean(-1,keepdim=True)+eps)*w

    def build_rope(self, sl):
        hd,th=self.cfg['head_dim'],self.cfg['rope_theta']
        pos=torch.arange(sl,device=self.device,dtype=torch.float32)
        freqs=1./(th**(torch.arange(0,hd,2,device=self.device,dtype=torch.float32)/hd))
        a=pos.unsqueeze(1)*freqs.unsqueeze(0); e=torch.cat([a,a],dim=-1); return e.cos(),e.sin()

    @staticmethod
    def apply_rope(x,cos,sin):
        s=x.shape[2]; c=cos[:s].unsqueeze(0).unsqueeze(0); sn=sin[:s].unsqueeze(0).unsqueeze(0)
        d2=x.shape[-1]//2; return x*c+torch.cat([-x[...,d2:],x[...,:d2]],dim=-1)*sn

    @staticmethod
    def repeat_kv(x,n):
        if n==1: return x
        b,nkv,s,hd=x.shape; return x.unsqueeze(2).expand(b,nkv,n,s,hd).reshape(b,nkv*n,s,hd)

    @torch.no_grad()
    def forward(self, input_ids):
        c=self.cfg; sl=input_ids.shape[1]; rc,rs=self.build_rope(sl); nr=c['n_heads']//c['n_kv_heads']
        h=self.get_weight(self.EMBED)[input_ids[0]].unsqueeze(0).float()
        for L in range(c['n_layers']):
            n=self.rms_norm(h,self.get_weight(self.ln(L,self.ATTN_NORM)),c['rms_eps'])
            q=self.linear(n,self.ln(L,self.ATTN_Q)).view(1,sl,c['n_heads'],c['head_dim']).transpose(1,2)
            k=self.linear(n,self.ln(L,self.ATTN_K)).view(1,sl,c['n_kv_heads'],c['head_dim']).transpose(1,2)
            v=self.linear(n,self.ln(L,self.ATTN_V)).view(1,sl,c['n_kv_heads'],c['head_dim']).transpose(1,2)
            q=self.rms_norm(q,self.get_weight(self.ln(L,'attn_q_norm.weight')),c['rms_eps'])
            k=self.rms_norm(k,self.get_weight(self.ln(L,'attn_k_norm.weight')),c['rms_eps'])
            q=self.apply_rope(q,rc,rs); k=self.apply_rope(k,rc,rs)
            k=self.repeat_kv(k,nr); v=self.repeat_kv(v,nr)
            a=torch.matmul(q,k.transpose(-2,-1))/math.sqrt(c['head_dim'])
            if sl>1: a=a.masked_fill(torch.triu(torch.ones(sl,sl,device=self.device),1).bool().unsqueeze(0).unsqueeze(0),float('-inf'))
            a=F.softmax(a,dim=-1); ao=torch.matmul(a,v).transpose(1,2).reshape(1,sl,c['hidden_dim'])
            h=h+self.linear(ao,self.ln(L,self.ATTN_O))
            nf=self.rms_norm(h,self.get_weight(self.ln(L,self.FFN_NORM)),c['rms_eps'])
            h=h+self.linear(F.silu(self.linear(nf,self.ln(L,self.FFN_GATE)))*self.linear(nf,self.ln(L,self.FFN_UP)),self.ln(L,self.FFN_DOWN))
            del q,k,v,a,ao,n,nf
            if not self.cache_dequant: torch.cuda.empty_cache()
        h=self.rms_norm(h,self.get_weight(self.NORM),c['rms_eps'])
        lm=self.get_weight(self.LM_HEAD)
        if lm is None: lm=self.get_weight(self.EMBED)
        return h@lm.t()

    def load_tokenizer(self):
        from transformers import AutoTokenizer
        self.tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-1.7B', trust_remote_code=True)

    def flip_group(self, layer, proj, group_idx):
        name=self.ln(layer,proj)
        if name not in self._unpacked_cache:
            w=self.weights[name]; of,inf=w['shape'][1],w['shape'][0]
            p,s=self.unpack_q1_0(w['raw'],of,inf); self._unpacked_cache[name]=(p,s,of,inf)
        self._unpacked_cache[name][0][group_idx]^=-1
        if name in self._dequant_cache: del self._dequant_cache[name]

    def generate(self, prompt, max_tokens=80):
        import re
        ids = self.tokenizer.encode(prompt, add_special_tokens=False)
        gen = list(ids); eos = self.tokenizer.eos_token_id; stop = {eos}
        try:
            im_end = self.tokenizer.encode('<|im_end|>', add_special_tokens=False)
            if len(im_end)==1: stop.add(im_end[0])
        except: pass
        with torch.no_grad():
            for _ in range(max_tokens):
                logits = self.forward(torch.tensor([gen], dtype=torch.long, device=self.device))
                nxt = logits[0,-1,:].argmax().item()
                if nxt in stop: break
                gen.append(nxt)
        out = self.tokenizer.decode(gen[len(ids):], skip_special_tokens=True)
        return re.sub(r'<think>.*?</think>', '', out, flags=re.DOTALL).strip()

In [ ]:
from huggingface_hub import hf_hub_download
model_path = './Bonsai-1.7B.gguf'
if not os.path.exists(model_path):
    hf_hub_download(repo_id='prism-ml/Bonsai-1.7B-gguf', filename='Bonsai-1.7B.gguf', local_dir='.')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg, weights = load_model(model_path, device)
engine = Q1Engine(cfg, weights, device)
engine.load_tokenizer()
print(f'Bonsai 1.7B loaded on {device}')

## The Task: Verbatim Text Extraction

We ask the model to extract a specific sentence from a passage: copy it exactly, nothing added, nothing changed. This is a common failure mode for small models, which tend to paraphrase, hallucinate, or start from the wrong sentence.

In [ ]:
TESTS = [
    {
        "name": "Legal clause",
        "system": "Extract the exact sentence from the passage that describes the liability limitation. Output ONLY the verbatim text, nothing else.",
        "prompt": 'Passage: "The agreement shall be governed by the laws of Delaware. Neither party shall be liable for consequential damages arising from performance under this agreement. All disputes shall be resolved through binding arbitration."\n\nExtract the sentence about liability limitation.',
        "target": "Neither party shall be liable for consequential damages arising from performance under this agreement.",
    },
    {
        "name": "Dosage instruction",
        "system": "Extract the exact sentence that states the recommended dosage. Output ONLY the verbatim text, nothing else.",
        "prompt": 'Prescribing information: "Administer orally once daily with food. The recommended starting dose is 10mg for adults over 18 years of age. Dose adjustments should be made at intervals of no less than 2 weeks."\n\nExtract the dosage sentence.',
        "target": "The recommended starting dose is 10mg for adults over 18 years of age.",
    },
    {
        "name": "Error message",
        "system": "Extract the exact error message from the log. Output ONLY the verbatim text, nothing else.",
        "prompt": 'Log output:\n[2026-04-14 03:22:15] INFO: Connection pool initialized with 50 workers\n[2026-04-14 03:22:18] ERROR: Failed to acquire lock on resource /db/users/schema — timeout after 30000ms\n[2026-04-14 03:22:18] INFO: Retrying with exponential backoff\n\nExtract the error message.',
        "target": "Failed to acquire lock on resource /db/users/schema \u2014 timeout after 30000ms",
    },
]

def run_test(engine, test):
    messages = [{'role':'system','content':test['system']},
                {'role':'user','content':test['prompt']}]
    try:
        prompt = engine.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except:
        prompt = f"{test['system']}\n\n{test['prompt']}\n\nAnswer:"
    output = engine.generate(prompt)
    exact = test['target'] in output
    return output, exact

## Step 1: The Base Model (before patching)

In [ ]:
print('Base Bonsai 1.7B verbatim extraction:\n')
base_results = {}
for test in TESTS:
    output, exact = run_test(engine, test)
    base_results[test['name']] = output
    status = 'EXACT' if exact else 'WRONG'
    print(f'[{status}] {test["name"]}')
    print(f'  Target:  {test["target"]}')
    print(f'  Output:  {output[:200]}')
    print()

## Step 2: The Patch

3,019 targeted sign flips found by lexicographic search (focused fitness).
Each flip toggles 128 binary weights in an FFN layer from +scale to -scale or vice versa.

In [ ]:
# Patch: 3,019 flips compressed to base64
PATCH_B64 = "eNp9vUmuZDGynrkXjR8EWm+2lgdBEFCSqiaqGqig7YvM46GM48nPYxgOv34a0mjN3/z7v8u//Yf/9t/+x3/+7//lf/7X//i//uv/89//7//5H/7NvMf+07/9u10+KwmT/ZlePptea/Znn7/5f/2//+t//POzXFPne3n5oi5t+ecXX59Frda/vvj//3//56OWNPqauLnitaTRdabXKPzN0dT452d/XUksGXhgKd39zwf2+rHoKfnn916fiTz3fbtIsRB8mNGyCt6QR4rDhbZaLbjQqpnzmd+etMkSvMEIhd/T/RoUvqc+Qp+NxOrrW2gdMXic2SuDXtHK9H/+2l9/cdza6CWs5XSN+1n+Y7XfrqT3Q/nr3b2fV0o0XKXvx/XXYvn7KmP+fsp/PxGbf1y/Xq8/Axa6z/L+5+t+X6J6FCwFF3se5O3ebCyK3nfXut+BryUJ2z9t0fY3iTLYBK3tQuHNc2hHjlQ23Hlox9COHJEQ/GL5Pz/6+8Yna66f2N6nheFrJTyS9gh6qa4atPf3K6W7zoq1aJ3sZVLXTaUrdV0/EaleFNBFNekh2l6V999y/cfVX1fkfsCLouH+iz1wa5X/eGfXI0La6d1Y1j923HUDTC2n9WMdiY95rX+8gsv+sHk+ugb7HdnuGy5WmV8/0Swt+Hsyq2gv6g4LeG9R+vdZ/A4nZp33S9mHI24332c4ZAwuOo6/FkOPqzRcaMuJ5HBm4z33wBYSSV+rMKeEYh/FQ3t8qqcwtPU/QqJfQ9uOpbA0M6b0et7uMzUUgmzuf3W98Z0QxVz/nk8PbZDaj+v+rb2SBy5CPIVSoR2Sh47vnXPa/ajtqr5HmnNy4PGWq+8pyxO3rsd9VcI1ePf9Uch+Sn5f+XvVGK3uqfhHZnH52t7XSdtzH9l0NOeOyE2JbSVEnlY+Q3cELHpUO3GNhF/b+zacPquIgM/cZSiXjL0KBHaS0I+ZuNL2y3idKK/NUk7f2hsiBpM7wZx8/O/48fqxSFg+Jq9K5b18Vv0j6bjvsoXhXXcCl9cMR2V4JdjOH+SeGI3MwrtOzMHatSju+Ap81y1PvLp+Fs9TvjzLyL+z9a8o507xe5dSEJR21Ddc//mk69ck2Z6E9poctfr9Ke/Ebta9/tp/z6FCSbd7giZi99BYT3y+hhD/O3T//TvWC2vK9UTA+zmtQm95p5BG5+2u1sIxs/e/mxp/35m26n3hL7W4p8ZaSu2A8B/ZULkMfdH2r+EBfuLE9VKyZ8U9LdvBFjeS0E2PadFVnMqWK8p8guO1yt41fXKLRJOiau+DPGkL7h2vmFHP35H676dihc2AXf1hpe3Lw+je9w28+ml/LxXDkkDmubnrz31K38sL2rX0PeHYz3g1nsrZ1DPaW6Pv2eFT5F1vuWw5hbnZtW3DKt/R0WF5WSw1jAmj9xdao2oYU9UMX6jVPW7tA9SgZNn1UQQ0QNqa6sJW6b6nj8NJsbqtgfTWZFEYmX3K40Id7YX5ueh9yXVxu87Nk3KRU5HQ7vV9tlJSWrafF3a1ntThGnplpd2PPHnqEr+1Ka0o2xvKUVLp1N0fFT5923GJGkKWmOKKiQ138p5297UrOnM/wHQoDdnxseveA6u475fcmSOF1F3XlWPD6unE3dtBkVgR7j0YsDHMsFGUn+LjElJL3fhCovqee7XjVEM/rcRLWbhz98b80GJhxN3VBxVITxV3e5+7Di5snYVDUDXzoG/to5Vaf/tZWWHH0ycgBkbSj+3a477RzIT6SjuHLQof1QtnGbvuCNqCOzULOkl2bpbYx9qH8j0klay/27V/r6u9naAlu2vywDxqP5GggGXl9WMWMAXtx6V0yOuKGcr3dlAqrOU1FC9F1tA9xK7zHRL1pYti534593hmYTSN6fRltMZ3lkWZlJUFDix0/R1Y3xtxrch7bN0nIWU3Hv3jYYnSqRD1TOb08kg078fkKTOgT/ipre8h15Qa8D04Zd4pRcGZbIuqXRx7atGVL8qkax+CsGv1ifn3t6+6uHNB3Rw9Tf7766hO2n6ZUfdHcaIYdvhzEtP2XVDZvT2uO29pellaRX/RLJ0Sx13H1L3h5GlO5cM++JvabDXuFAZOiIMKLfaawr50OoRGFS2avEpSjr0fiAVX0OO0nye4bqpWpyLHRBsbbYJ9qnmaCpc9s49xeqPxedvXfEg/0fZSOOkz7riCQ/a2Djxk9G8sxzuLFZuC3VFCKcrOr1x47jfQg6sfOyC94LbngcvcU70DuYAKtHKoyJeUBVvKrBqmvC5YT3VZ4Ej2ma1eZ/r77MHcvaWwyA/xufcfVRLXf2j/mD8KhZKuwU7Xzi6D7nwfGmbY3VgLZ+bzVLXXp6JP8+xakS2j89Um/s48v6L5CPaTJ95dqe/ZZOA22PeQi9sLsvA395EalFS5ijv0Hp71clkSOyLeE+gzXeDOp1M9uhM0VwwNaxQPVZv5EdQRVbbjfSeAp4JL410DGG69bIeCaR4c4T24Fa+V7HWHjng34qN6XoCTf5l9Yxt5RBHctvOaxa31jLoXs6fTd6+nMlSb36oiXOv0EigG7GQJkmjdxee9FtkpOaJRYn8JWyh/HvMFXdFVFIr29vB7L2eXuYmzlxhF5NS+yIYpkBaWnp3hXI4rY/NszShi80zuGImTUAN6Yp8hlERpv5fd33/whEM8elb8XZC/OkeBnfqM+RGa9x4GPFBNUxNCB1MG26V6UhUspVhUedR9gGRewu97JxvYn9N33fLu8S/Ei9ITOQAWxuX6UkzZFgRlT7138SO8KfONNdBWj7P6MQ7qJ7u9rv792qBk3EUoJJwHmMEj16FjdhdA676K9dMRtgtA4T0m+e4wqyKcyn8kEEsE/+oBuRCkJnaFgf1df8ZA9zlEC5Z4FnjmR2UbhRnCzexTHW9uVv49rv3K80zxFK6/YUZ/v6DX+vlqFTZEmNV5H6HvQrKx4+CeiD7KfsKn38EqC0dVWdwG3TkorNedt0Mc32F1cG+0qtHkwwlrsw9fx2v0D2XgUkvu8hof5blKwzCYf4OMXmHrScqv/QFZxRDMknt7zKTDcUDaAQgSHhSqFoJj5TPU0/vsa2jAqDtaU6eoTYKuZYcuxCFnPfj3+/FllCX0ruAQ3V/Kf3I6sOj4w8S57O1yejs7SNJ+24srB7s3KvScZz9mDK4VwUhqcYuGgZwHTdY8uu95/hnyQqOXQK4SKdQW3FUWtQd2hpM4GzYGesenH37HI7wYJe+1XBX3zGMy+57RHrIMPN19blMY2TmrQ15XVknpmSWyo1xxSDFZ+gOhlNjMXWV3KHrsjc3UKIeBm4zh5GxXSk75wV6K9LCiizaa7QKXTo2dqDhCv3eIv/eid16ngmGJGkQlDxbkhhx7zV++8tJsrFRz1X2Z7r8HYBt5k6xep0wJvcvcy5T62rIEWr8WPjjVW4ONwklY+CdHoVQ2ehE4faf92IIOz8Y+p9I5rgi0PDSX+yjLbQHbaadQnIvunNjv/SDpUkJAiYoQqj2eGcX1oO7AXmvkp0d2Qd6LUHq7s6R2HBIVNDF2BVGOZAjO2HQ9vaA7dfKBzdzLP3sjQV57aeG8RO2DE72cZ59q5hps+zVd+peGKcyERRIbuiugCWaZdn/IhRBj7Rfe9p3TLzj+TCim70QNyud0H8ZJVCHsXmpg4BQvEN/7OiaXIR5swQj6hAGGTblCt+FsQGikL0VAhkQEgCSL2C0q91cyOzwYMhosud+9kjdYGQ5b82EpwawyAci506PErNts4aBmF/CG5HJTv/MFvCeYiiLITthhWKgruWvPAWr8gwiUOyR/QXc6yrBR1baQrDqteKiaqGFeKEPh6HBYGGJm0LoTU7uPAKa67yt8V2YLWzy76mFOTwumtYf5pXj8rKE07qxX6NjvJ4JU3JVrQQ5iek+GM3Rx/1ZWQC600/y51xtVcOlaz8O4vK98Cs5LsNIXbOS9Nnb4prN9p6bcJpMHGXqFU/cs7CLvA4Enz72zaBy4eYgUsoiNZ407E0lqSu4yHTJ6n46AoJyBTX5u1u88FZFuUsgBjf3Kg/fUe47/ldkEILh3NSuUhOfOAoCmtX703lcK1Z6l+FGbIUbBPzDze9N7Idh2l/eN3ZU+OEWoCF/Upe+2iwnt4Sbmw17MjCD1ximqecOZL/sPYh6z0xieVu+KgXE640/JcB/tjA4ig3kjzBIn/uBe0oSf0RlMj1cqXog1vJ/cL1xR4kEcMtDCxuupDhvrmpkfbFFMrrMCFqVQ17uL5/o2ZTj7sIcremP1+TgVQsRzyfVQie7CG6IIgQx+LbNcaCfOC8L5FXojC5rvhiP/vQaI9e5VhpCyFTxq9Alsh/cMCyMYVvX5ufHr8wpfhKu0LsC3pYdSc7JF6aTq6eD2WBYQYcZ9eJzYCl2JKYdOlz2t1Xsd6CgG0fEDlLQzzXCEmwnKWfmPobTs1BZw0ZY/5qGTfh8UZ2A+KT0YIsNh9uHLkzqXRwWBz5ucBSzgcl7kJos2vvUKXKxG05RduXkyZzpQ4sqRBJnnDgiTeADJiLYaoTyuu6EpFKmGkP/2gaI7uhFbtItPw126yzfqk/VCqviIMZtXFmPkd7lYSEj+8HauU6HWQIrWgXHyxbTfQfkyT7vhqhrUOM3euwrkmSyMgsLOo7HIlFJswXSuvpeZmlbDfEcche3DT+lB7gWbVAnY9BoMhzQV0MaOX2tRh1Y+TaIrS/iBfUM2g+lHVQEiv1djqu/yTOOvK9kXRLy9uY1SWt9HPgYM16SjSqQbJUvCAwkMrYiU29eCS8Et/D4itF083PPTEMMUzz8U6Pv51mVFsl/t8NqysKuZOcTx9wUcuBed6ltZBzXj/Ekmr5QfFAxwe+Skrt006DPEzluR36y9YFS9QxnyUGsV6j+Eyh0kcBgGHNwH5RgoaIo+ck3XqNmKDeHOFK553CuwI0Awh7LAM1TlxWj7Qky5Ak51kipRezfsX+mxLNzO5asojCUCUXcFiz24nYEZAsZ3ZCHgzaGKY13zA4WYqwpgBGqFMeKAwhkjN4ZRbh+GHoD9aKD376zaikH0A+fWQdmCfoJ7APYvE/lD5hILdQYa1WlyB2JaC+FDRLK9aQxlI2zFYvYtqmnIn1HE9bU1AEoqk/FLHYUSQFYwmAx/OAxXGIq2O0SE1fdi9ERBlM3MRHW6XB9m2rVvXShGO39QfND4LfqjO15j4zQ/Sn6XyYK9em9fx6Ew/eQoAIEIyjTJR+xyQTEDe9YkpOKZODAMJmLsOzCGBSJleKcBIih9NMhOFynQnDy9PphqRmcDgxR1gvMj/HV/M1UoBlzZf4+L3rtjHFUddlBL7uYrCtS0G8XQcsGNdQYjSB7v5xauqLD1cD7vxPKXpul7fZkAwDIXjNd2MVesQrdq0XztqE3f61TLV+h9sWeCJ/vn1QQwiojMuoO13LP+va0Nc3tdKMrWO3FCDsAJCcSwWgP6XToB8qN/BJjuM2XFLtRh8JIW4UIW3uxwQIfzad46DucNlXlPJTeoxbWwDaX7AAY+y745HOmaA8JVVyF3wPaa+8F1fDMZvjRgrClPqNXC1VxCpb2fZKImiA3bCpiqYvVw4G93Ckc38f4kVHmwb6yN3AW4J9UowsNHojTofiILulp/rBSu42HRdU9Q97NP1icZBtgfXAVqDwgLQEpG0gnXOX1vCp/pI5UAI8G1yj5wEDZXhkdfhoEg5n7dXP1oP/Jgl+dcims5/0DWLt96WMlyB0iiVMOuR37kVevROL885QMEohI5Zgh6m6Os1rBjfTrrgwetvqoEepZ8pt/3JMIRUlr5Qhd9AV+LQb2yBjMM9UwmSa/EtEUakgzb2SFB3g5/kXIk3wkzHYJHsAxxYTsGEA7NfxgZxCvP+GqcvYYx7/s+x8/9NNvZBIl0ayglE7OKOYrC8KS9IRfMybQXcAnWj0bnTs+TjREKgZF/YFLXk8yc0cxs/yOpKMR0dFCoA7Xrkntj8Og04ARqykkxa61kWdhyZRrsQm3S0AQVB/9A86/bY2d0OKRZrTgS2pm2URkU+qRL17XnL87h18BrBPUfHPPmfVbnPevveLUOvwxmahG4IUeYfqo/SsqlDOv/qFDfc4pk+6n2xanBD7nmfhR67zjDfrp2NDaCB7PTcWjFilkh4iYFBMLXMPhtAode1tOFWvY+g5iVRljELm45YX1ENK8Pa5eBNNFzyx9MTMUpW+1MvXB2TYCQbFFUBQ4ecp6stHDKTGoUvhYAv1XfFJVX4haB2XjIDF1/qnEzI3XBuIyQwn8Uwq9J4hIod9RYXMFbSMw73k5oX2eEABbtDMjvd7X/H1KkXZvbfcqziyfqf6cEMuZlEAR4mNn31k1ngSnEPhYAP14qBnpBYTgCbB/0yjITRQpmLqD2qi6kQq36BWxmuE8ulGGoCWG2gHGSZrSyrV6eOF9yHbWMa5V0FigFAFekLmzdlNQCB54kMKIsYTXAjkERsf0ZaySqqnK+q77IlqgAh7IrCsRu7p9CQkbGcAhRUUPob0ZDcWritH93pWiQsDtrLu/nARqbdoxbUJd7Yck67+rgW4FrsBb8lDd6l5BXPj9X3pVxGxeeCNpEHp0JSj1tPUndZfiWL/DKK4AvRNxWFMiGzMfA7K7IoSBRtXNOBKh3O8ta7/K9sHrMRB0kBTXYiFeL7nuCyXyFHVkxbR6dpgNg/yAD5CxfzbFv3P4Ia1EpyRdls0cIesjNC2HxBTBbbA71kX6D060AQjrsFhBt3Cf9sLuuIQ0nzN0hP5iU0pxtKxLkD9+QntdH59luLsPQ55U3iPqrIjBnzXFLT260O3UuKmB/WH1i3aXz7eySutcBQCVtYSJwpmSsfbojBhoiLSMHoHwYkdfhjy1By4NnXnHXiFlVuEc/c4Krx7ME+luv5cT9kGYStz00mdswJqjb55+6/dobSi9mP9WQHJIFf22v13JuWAJYrloX4lhWgDSHaiN8V8cRVbhTQqfWgiSKM05yAd6hAGhqb0QLTzvWZDuBI6xTdaBmhrkHpU2Il29ZRglvffRtb6DTGpYSMgS4SosQxxsxb7ozkkA5igUCYvsUjUbscQK9choTEkUlgo+uErAdo0hwIO+vSyKTxzZ7daM/ZARqreQHInRZpfNw2q9/cR8KONf48sj92u/j+D61EHl05j10mcX4LvvMLsDscZgo1mAEeVjZQPuzV+/m1d5rFLbe71RRf/tjxwuqe+bJot7oZnKo8oRrq7fUwtfFKDLPdo2bOC6RJUmwtkBRLHmGHvdp1UDbdtdlUCEe1j7uxb2t0D81WK+8iwvq83voIqXOLYHxBXWblAKYb6cVYI4ZOQRb2M+DZA+P4x0m+K+h5VcyMlg+ir/QJq/YipwnGWyYZ6NiVn3axleSxcde7Fo97tqcAsnxV8cdvLNIllhU7rTLWoksh+VWDMFpLrLCGzNCbEzILokIPrXr+0H+khQawRQ7EuVK1kZocTAK3OvfHU14i2X+d1GHiY798UW9Ozj1j17nPq+Sc+wosNgufC470ycewpF+QG67sOTR6ELGtrWCH4zsLPR+ls2ghUnp4zNlgJcjpE2znPE5bJHFp4/HIFCCF9pXZmBpJBLMXT71A+6fJ3Zcr7MMrAZVE9y8UgNi/ZmK0BHdbxO5f3Gv5A/z3ef6qvoSpHDOciVkgglbGx1zSOr3mywkC+3tD7wlNx1Ohgp3wD6kx3HEWEjkO40gYayvLfrB1mYd37EXCux9f9zV0SykS59xutMUWIGKc2zaFjiPDqCf05InCBGD+joWk2yhM4uExlYvtpcKAAOcnj4aeSTljjLh6KSSCOqxj6/2nQltU1RfSwzJue21nAj+6icpuqrANWL1lT06VdCvWk5ef39cwaSanfA7tcCkfvjC7jMCYSHjOpQYK9rytZhD52kx5NXlUVa4Zwv7RIIGwkcp6m4KypyzYxdUIJQ2ys0AMxTVrD/TykvPORUCvYXjK2Um+k44uSt4mv7Diwsle6US7ZtzZ0KL4AQo3HzMm9laI53Y0GMLm3/hilZ7wg4yZ58uBAlbAtpaXl3gr8Zy/DDh7UdM/8reywQe7Tk0Ic6kG9krpbFR6n4mKDXh5TgSiNZy5HklwEV3CCWtg/hAju/69dz2z0rU4e1Iln6Z+qHcsRMvKlaP6jbKJ8uLf/KVUDcFlBHEOP7pFfgte0ef+JSFnk0HsmtQ5OVa5GtR1BDIeszH9F4osASkFpPa42Ge3MUEB8mOh3bDZPglyocL5WqHwwq52vE8QtHuUQbJi4Pnkbg5Yj5NkV0c05jSHGW2QFtMT8I2RWiiSjhXCiepBGSnFGrSzz7dA+U5ZHH9tH8N9a01aVt5veDBXxCTR6TvPgB2oETFIyIOqG+yGN/PBPX9QyoTaWKRvNA/Ehm3fWyvmexXv8KUcihsB+YaLAYqMrC+2Bk6rfR2U6ckvEAxJFzgzKzC9s0R0MfEd1dpAjJrBQ701kzn6v5x4Oy9BmYDB+XKInEruPUmb5bti9YokTxfEtpQS30hHcJ/iKI543w6wbBzPjd+wTGEM7nyDEsQ8rxeppZfkPpmfbyC1Mp2SYJ2q/6cRZfBqgdKudeRtf+xZ1YCeK+dFNE0C6FlIsHgAcv4IRnrzm2+H+K8Rmiwkxs6OMMHQUykYIS6k0mjw/4Pnu4aO4UC/E63qBmH6vDzUYK67pgQyAJ2dRO0PrKfUf5NZry1UPGPPR1VakDaW9GD2Ma5F2rm6AXWhRObY0CHQ72Ppvwlq/ggcuWmu87Mrhbya7I3L+I1yRymCCg7Mu2KwfFU8CRqwf45Go1WLpTbPUReVg9aBrRb2VUzjhCA8LH/H7mSe/81al/nS+v4i5aGGJwxesKWZZgOZQuiiPb3muQc6gA0KF/9EEyAkRcgVH9W0JBmIkSDnY2COM0xgUQwU7HFrnVwR1uDvDWiwhyV18wJBcid2DMXZYtaUQd/to+2KcFqGzQL0GjkmJ+xwn41wnua8mwrg37kU8vcE4YXy/brIlBCe2fLQUQeMYBr6Ecp9Iqj1EDx548E7N1Kq5PMSrN+tJfBO+ojECk37JwbWmyNsTnRCF38vJ/9N3AKW7pp6VhPWaKW9EeL+abzaINQgWUJrDYb1FLZuRhat8leV4nNs1Bs3qQpg3rmLZ78MoSLwdMOJYGOcXCiEQSg6GsEWQ7S3QPk4McG8S4HYcVG4sfNDKX3Ptpj94IckI8iKVjblQYSg+bjsHAXKvd0nhU+apGX+YTUDyrA/CDzt39gKldXYU9KOfaGhB5475T/vgtkvY3eX1M4vIq938Cayo+dLM8LUNR6p79N9a67MyzpQA8at2rAWMNWQQ2kyoQdM6hOd/7LNeHEQqb+/jFHHdUWFIk6Cgw/gO/ssHtk6wfSSPX7XHJXd/ev7PSAWqHuPyJhi84dmZ9mGcilQm/dTJZg31U+Znz7OaEy8oG93/fMjscGsbrih5fGQobeLBbwqI995022iN29dT9iBdqoLfKZYYTf6SdDjliP8+RdGCahC5Q/ZmEtL2nG1/OVJdzWXo4zlULHsJ1yiOEpigJ0X5ox7xACJujSBRqvOwoguV27FT/Lpzt3NSSxtX6oCTa7jLKqcvXyhciMwjyy6uFB3g8TM3To3kcGGNp8HIXBsEo1QIvw0Wm5H+bZ0BdJd2Rz2UdBW+4+Pob+7z0lyIetua8U+RBAbzPrtVDcdvEJ+kL2fFW9zNXXTgVghijJAuyHX4wERi/DPD70jPftYp/DZDXsWmir7sZYuh3GmeC2E1MUXW5BM+tjOORoVJTa+NkDwLjJey4EO2aoIt5mXyS39J1l8ycMOV373BZ6fyUPGeNOo4lFE84dmguPU1+MIkmXZIAlti3qrfv71ZuTqB8j9FDuSS70cYVJ2k6dBIbrMpx+HP8XBxPULDSeW+1scvTshHvv2Ukp5TMYuuOSm5seGTHJujEmIAzvC6EP7YPg9tirhHgZstjdZqd5Bf2j5bHYWTVQtKcVhHVzJJRshVFA+UDZmogqDudm6GBnPaWRqjH75GSltQGb5ephDfcfopv7t0AsomKWY6t+giXYGhyVPQRBqi2yaPF0Pl3ra+CiSuuQc5ka6Cl4zKk1qR5+zMDtXlUzFsezBauVZlBK6qMQfQt3vVBTZ1S5fs7HjfKuYOjcisi9Qege9vdIKfIDNb9tHaUiM9bLMe3rUc4CxZF5UomrO5vgSF6TaJ07uTWskXWAZbxPbkXSxPqBRK34ofOkPYZ6dZ4ojrOCGCj9OmS/4LkBghY7T0DJtuMEWuwg2uCpV4t1kkuL8yfPYuCPpMd9ruNO5UUvls84viEEk5ry+5yxIgCM2fnSlP3uYpFV+kc+9c5mkWIR8ahkKEsaMhw+Dq63EYIWyi7q2xH6lVs4m86EFMgO7DMWzZPsj3rfHV1SCJs4XhcLBfEf9sltQDnjxHSdYrUawSROP346l5pkHzigvtUlxCWSMbAikOIuORtB7D9IW2ZO+5Gq/B1GYKIv5oUAc+hY9rEWgo8+snp3WuUQ43IUOw2aI8M9cvbCqn4gn3K35VrMwpMYBTh+09M6K25Bovv0Yu9egIk+svsjDseOeD7vp9S4NyeDDTFbQJ4q07HV81Fhh+doQCizRtRz/GrEiSML5HShgcq82A6hVudAf3oE/Rb1hztGj2OUI4Bj+Is0/Q28Q/pBqxGaW7sW6AcVyEVlSCHMgqClvRxAxDvtx03dVb82rgjjVZZjJljLuJcZ3F/MqftCPd6rP8AsVcj9kMDU/tgksBa6JLo3qUb3D2t7+j2wPN6VLwor+iP5DxsUHBJrIkh8zWDkeroRyB7AQJA1g518EcT1ijZTMNsSMRWywlF1Rb1p2eUCy1ApgDdaLcAzfdRZ7ymxsay40dxGRtBDWyiRPlhslCr6eKkAv1iKt7wJ6oW5aWOhYzGOPG7DaexbnuzF9fw0rO94EMfegU7hsVuCs5JjQ20goS02aLUuyUJMXfdKp9Y0OidWOhPIAm2u8mPBe3sBn6btfVG28WIOFnob8/vh5R9/7bud6mCWdWb9DsC9YpRLevwY4GGs0L0mCde3T0tHLKw8Ufxabs6A0u0B2DuorCl1uJMj3b52ROtOLIPS2xw9aXIJelBHo1B/SdSwG09STlQJmoqdKDK86yOlWZtbIVn7zJsAdCCv1/L1opdA4F0DxBn5NERudJX4Uea4KdOTXPGAVZsEc8AACYm2pRBwO4PaLyfBQiBmBzrf6geRQu5ctijlL5qT11v15Iv+bNhsOG4m6BQyDe+6dwhH8/I3Ne8r9icL/Oy60GnYnLmYoeQv6fcvyLyyxdN+56iQLssRkCwTamAEkGpgfEEAsBgTCrn1CdUg8/zYo9zCxXuy+DYPQNcYE+qX6w51d7WdU6wFuiRTyA3NUURuMOC1ZUGPena1AJ5cIwIJa73F997a82Acsji57DBSz5wCc8IuxbRdWJA1UVlYa6HZm65ES6rTPqRd5LECnm2AUeUugAzFa6uZmpkTKJ9w+hOUQ+0Xgx5RpcVGQp1wOO2YUzhtG+dazH8QJsOR8p9SqOfz53v3E6NhKC5pRT58ySzdYMe86rpHsfmoYN3LC1+o6ZHtDDzI9HvP7KBsqHdTO63H7l0sYsj8ETW9UlqO3jJ6ViPv1HPYr0nScRMflw3C54d7Evjj4UBczzP/iH/d4OsxKNNPxDGfAnKsfnK664xxlrPObvSCvG1HYyx1RB0TLesGdv6uGZNOoHFBPOXkM2++jmx35kkhdEd6pOj3YhHIRIH8fMvSv2cmwPXyVBA2C1ddyPXSMnTDk8WMuRpO4cuqiOl4XDtRdqRe1i/fSmrEOq8P/fMa1lBVnIVdDhsG2vnWWKolu4c5e9PpPqNw/HhMw0mFRWcB8sDKGWWyEsUYjmob9XFzjaN/qxWDFJMRTlEL83f3VY7OnM/U/C7Ks5CA1n8U6+69/UGn794JFEEadrXKc2fLCIoa2mzdqZ6CeUZOG4KBJxCYsQM+YNR6iSG3d71waN8oWsdgaisQ5Dg6qCZVPTyBlkJZwVkokGRrAq0darBDFBZoYdvphvfw+I3cJbcp1Tgqq/dE/GOVeU3M+hd02Nj3vQytnBjWvtManKod//DEsU3LD4F7lrFJMyQuPSJa91JisPI6cjoJHBMD36J97iBN1tA2Obx+4KCcNTDPkIubY0uVXMKV7akdZSn3R/japIKCybg7e6otmr6bBAh2nQT3DhOJ5JW8Azr3rfUlsf8vOvXAH1+G0rF6rH3A35IFzESpYePqM6gyIbzXmkmTAkmxfgSlb+nTeuW2b/OAamS/06gn3mL4330o1oOIvarAnSe7MIT7PhWSZ0SEUN4FyyI3viTr7NFkEawjDwfBZUXQ+vBppIbtKI7+GOLpBrtMoVW5I9n8cF7PxfqiDciNWkannXzgUzd1/EZ81F4FRm3rzBwmHw2E9ljRhNLtQffZrmKx92Rqu66E/bRreOeZiC32F92pFhKk9vMHL5FdGIHJUIgDCzMG4cct3dAcrylSFrBVqAhpCK/rMNqd+bjpgZPjIAxnv5U7tK6yACJyhFtpfezsEBKGKNS5tX2aIU2+mD2Xms6yJvmCt76TkOFsbsXCM2vd++q+U4LF1n0p9GvzlgT/Qr+0GliXEWBs13mcU+axLqNWbCUq34xhSTYtXHU9qe9dl8U49WIZlX1eUGJvo7i6yuaHskwCG6oyZMBvgaZhVgsmLx4sgSk+qBSl87JUfD/F4UT0Y3oGDjqj98A/Qpj8yXH5YQaLMnXV8qOTiZi2SSfSjEyCu8y+L/bN2kkIimoumvXtLxldhaFsT77hhN/a6DjSjo9w781gQtB/bKe9mB9Won9y/rGYv75R6VFGg65GvzMH8KAelyVUVWq9nxm+Q+Di+y4s5QW9kI9OJx6vZUo58y484j7bnSwmqXlBJdyZQxNXtIMvYPFWOzcwtX64cBmOQvarZFTsjoANOZSy3YlO2j1ySif09mVeTdYXYjNZhqBjEC9jZcT1ztUoQz6LzzrLCbRqWkY9VF/FnDft9xDohdFhC8q9ZfDg9Z36IO6uYfK1izScBXo1ewbsAlpAdUIV5yv2qltfmIzCZrQfbsk9WfpIB92y/UYm5T7uUPCsSkEO0pc6GrWktUKQe4GqvnreM1jqxiLMhnBr68wJQNXrIQrdOdlFlC8/FRoWfT5F2uS2S/xGW0FRQGP3D+mRxYdk/5CvqQ+U5uqvqpieo7jDrsSE8vMPqgF60/g8IpHlfZQFCLwYe1M3iuHMgDzNpyt2T0iRtbFXghC3LkA1e+92sv2ymeaOWaLmyOh7rX77uRnm0yr6y1YE6YG9QLRvLKEB4CX1g5g5OoBzGyQE7eDIA+3Ggj3lHg7CnFlap7YDEeMQbyCWOOGAfVfdBtlEAy1DLJCFtQ+DQIaiJiRRKVYEc7eP0N1VbGDQJOJIqjVx/x+Syt3dw4vydgtq25kYqZ3KrqrvyetehkiPmxQOmqbm/GFDQI0gbfd8WXV+dRIUfTdzCbfOPV4ebi8eGWsIxS4I77nEEeRzNBt8UcXflmo/xH5KaQIxGkUIs/CXQdU3sDHIH3hvNEGroDbgRrsshFyoBVTPR7VkkMdfTnHYH0XBu5NqsxalOJQQIboQu2iG3gz7daMNzWEvkibLUc1wYMGnC742IjkfxDGPFbUhrzxCcoK+oCqMSP14nF0lB5XOn7BHvlMAQD4/2N2LRfTsBzp6VBP7iInj7ljFkJfRFzz2K2UShGgcQ2uYHbzwAV/0AMfhku5qmXAd1Z/x6QXiLTip2IGBtCfCHxzZjfmQ0ggKfnTsr430F5Ts6/kb6OWePJd9S5JmH7aKVLJ8sK99ECSFIxPjmB0jJjDxWQgFlR3O64fXK/ao08sLgSkNGu2+9wYKkXpAvrVz2XTMgcbxbc87P/qWb7+fjNlGEk67qMHDY+9fZyxIPjjFe0tLOX1+rDxvtr8OfleB5gMq5ujJtV81Tnm/NJC+iQICJr3umA20FqgLHfoyMkzfN/BCjXtTR3P1oJDUi9T5LyMCSu8iai22I6HluCs55JLMG0b8XiCpgaY0yylzOrkfv7bF4GrzWYGTE01FQpYNQii9Qwh6ANTmHAEhDFfBUyg/La3beWKm2BPvieID0Vhy/6NBdYdilGEX5NNkuAYuZ/2VMJjm7ToxEtHazZOy035Fj0JRLC6sn1T6ms102MA4TzmGSiiVmLKmQFAiikdb+QPzu8+pH/LRw6qC8za1+zoHYgl23WshcyoFVWmltalDZ5mN/kdfLjsvZIaiQEP5a0t+HQaiELl3eMN4E62NqcvRRECyXCTql6qQIXJ9VE/v7ce3zO1b2gZZPfaojV6yE1/IDlEpDPm5ayvqWu7MMbEjmEF0LOF6JiqJg9PLmymOhvyiaM67nCLRUdFRvkiFcY8NGiN0N9qF+CRAH3cmwd5ONSg7oRGJ8x4LRugqUdvLBsqmIzIu5HSOKp26i1MQYWo2J+ldfXI/6DMIvTM3He09ZJRphft4d2wIDVclO1MdjgiRZBG7FD0QTIkWPvko8V6ba/EIMt91LqRpFur9bMWbg2wXSumLo1mjrJce4bdLBkrgtv+QsdkXg0X7PgXsfgv7hEOS1K7LhwqQo3pGCXA0ta58KZiL2Zea8NeanQTYxGI6XHwsZO/EQsTPLkEZD9duQnrby2r+XfoZpMUdwbvNCurd1dhs3FWm3GNTaGQzqTATeDBv974vWMqjX33L97uJMeQfRcX7gWKKNPbTGkQNtcQxjg+5Yfmxm+TJhHMo3BHbSOerUYJnL35qIE+8xI6/YSE42a+chShkQWUTb5xbTLBui7mSZOtOZbCH+kGx31S5AvURv6Smv0g3ZvflKtPowGBGtPhdEsJeG3tbRr12xqCIef2gGKoCxkS0k9wVDDFUKQvdEF1ioFJvBOPIjwadK0lS2aNwcH1OaZAjmi10iobkXdUVzZEDAdlHLPT+KE5ajETEfsG4v4iPoE6e0o0Cb/vYhwe46xXAaSeLIlpHJXuCo1GUrcSu15HcQfB0OLclJEQohdJdsxSLKaKM2zRSk9qBktJrcPaya2GWcqlieans1TCY/TCC7mVhOGoc7m12P5eOUPTCAnUNbXf5lDr3cag7Yvbs6XLclflm0bAqdhCBsFmGLqBigeCZA1QDKlQGeL1JJboBuDVu4B1Ukdq2z37FGFgyDE3FZ7xepmFfG+A1o/sWuEXTlBnmlJ1KDjeqPIJ5wIaqwEU0JYiW1vcf/bocKVRiOJLMlBatZ1J888z+9BdutyCDh9HiHwslcXxblug8/plL3Se9KWhcNaK4sdyw1RyZygl5Gbt6ODqWtA0rdlixt3dW4/mz9yuIXEVJonSL6w9jQWPRlHn3cl4155Aerq9H6ekurt+Bfe9dbqCOoq1qJKc/lIIb4i2TD1Z3Wkb7JCczwo5e3N7NdnSRNEZ1yGeJ3crc15985cOGvpq6DODH+/p5Cm7pBtpdb4fjbxauALHaUHhfVg8IdB7wMWqneaIweZHRe3+gZjdQ+EuS9HsXcuitx5kNlGCmAxXGVrN70HQz1iZR8ST17ZX66kM51Rb9ckz+8ldtFGXZQeaHUPFAKTgLCwz/gfTPZq02WcXTp+NKTwiXXAsnRRXq/F4fg6Q7D9NQY3oHWFwPZwhDJ2fm+mEX8UI7vtovgta+ET+43I3yJBoLiJh7OYM87KmvUGNpF94owF7Kk8oqFWo6NavKrijUK5bx/mFpjIDG9RxHVzP1YD3DrtXss+ocVI4JMdtNQckTx6j33iTYpwoUGcs41B+nYZQJTEUK0OHAJRp9hKDw7R8i5L0xgWo1GYtFFx1Fw8pnUefsCIvAqKkVcWzeLDKku0BZaHXzKtG/UFQabNWZbN0hIwj02RknuB+mNpiqHW4UVjy+lxiOfl5ukF81jWLdOcZGbW1Br+cwdxFLeC6TUtURYbCzYj1nYSDL9JWOvonTYGRROT7QlQ0HqzhTpCvlTjKK/Fu6sOxQJFEemWm4QulhG+EVaAesuah1XI+d+d0ewnrYgOaF0PuXxPc+38nPVPpaT34gJ5cO/HppzL4L7AbLtIWqbiqOHdizOzGaCZQO2o9Ihd5cThbiEnIlLqtdg/U9ITmNJIbrZiilrl7cbDFfygPpxucVo7xE4sOAuYvuurDh8iM7fk0wdpAMtH5/gA3X48ZLhAjGi4GygZx87ZeL3hdVGK0UU8ecGbDODunTwgYq4hCEsPOx33hQvDgWe83jBkW97R3ygtW2FINocNrYqazP0Fk10KqYBdy3ZabIFTFMunYxhm3ajlEesxHL/qv9+S8VMS50V7tP8WsClKBG7Yf7VA+aJeztAQL3+g4a3xISKAIzBytHJ5UtY9KTDnUkXhXQN+sJDuiDCwNjyvWs/+vo+VPv3iV2ZLGWqAddY/gDYQHFPK37ubi/ZSiMaDH8Sj/ua3AUMBm2tRg3mYvhVQfwS8WfHKwj4BQCz1Xy0FqKittZC9qCkRI4GR7WwO52HCbuJZQoaYG4E28490+JDETT9UMkvV5z8ldg6gxsrrahloGhYeGuebBMlE5D0aqnu6pXJWs0L1sLy4nsaaR+hiRIIPmQlkGNQ0sgDVWOVUWRdLLLwzt3PEyEOBR1NFvu88WE9i/jNtVR0XtSDAuNkhqeAuyEHJt6rYouHr1+CJQUFg07HqWBRbkWiKtMxT3epKMYhCTrAM6Lpv6FVa1wpE/QyaTxYmh9pfhDqiY7Q0Tp853806m1C1Vs74gIiVZo+g87OUE6Rv7Rebmc48OzCmn7ofQ7uHp29tiB/WnC9pxxL8TNs+UJoLUYHscyB7Iy0HNwLdjY+9Bs6vruqgW1ye3jNnOl/GOhdyYYKNOgL5W6r2wifiCdB9Xjj6jfPT5WvRRDvhCP7zbyN5wZ4Z4/NGynXrOPl02K4NbeNfMPL5QX4vdbmSpgwKQfUfnL+c8udjs8glz40eM08N9JEE3Mes29vzJ+F0FEssDbVPk0ye410AwG1h6BA9QHvUt7Zz2o05CJOlgeLzm670oNj/kjfssttF58biwTJFIGiubqFDr+uRk6qe6sE39P10oW9rf8cTQr01aisKv+wzpp3zp1eORocFCvZmlQrnPMhh2jV4PC9iFuobvFfj8gjTrPFO9+HUYRzwe5eKUI8TiIEgR6RSGXaqd+4Ah3YBCQS6oibz+qkEGqR8nkHoRc2cPpEdu7q+YXV9I7tQ5mF9D84QAW6Vjc60O5lBu5V5sVjZHGVuI8TewEy//0vwF+JOXE"

# Decode
flips = json.loads(zlib.decompress(base64.b64decode(PATCH_B64)))
print(f'Patch: {len(flips)} sign flips')
print(f'Compressed size: {len(PATCH_B64)} chars ({len(base64.b64decode(PATCH_B64)):,} bytes)')

# Model stats
total_weights = sum(
    w['shape'][0] * (w['shape'][1] if len(w['shape'])>1 else 1)
    for w in engine.weights.values()
    if w['type'] in ('Q1_0','q1_0','41')
)
bits_changed = len(flips) * 128
print(f'Weights changed: {bits_changed:,} / {total_weights:,} ({100*bits_changed/total_weights:.5f}%)')

## Step 3: Apply the Patch

In [ ]:
t0 = time.time()
for layer, proj, group in flips:
    engine.flip_group(layer, proj, group)
elapsed = time.time() - t0
print(f'Applied {len(flips)} flips in {elapsed*1000:.1f}ms')

## Step 4: The Patched Model

In [ ]:
print('Patched Bonsai 1.7B verbatim extraction:\n')
for test in TESTS:
    output, exact = run_test(engine, test)
    base_out = base_results[test['name']]
    changed = output != base_out
    status = 'EXACT' if exact else 'WRONG'
    delta = ' CHANGED' if changed else ''
    print(f'[{status}{delta}] {test["name"]}')
    print(f'  Target:  {test["target"]}')
    if changed:
        print(f'  Before:  {base_out[:200]}')
        print(f'  After:   {output[:200]}')
    else:
        print(f'  Output:  {output[:200]}')
    print()

## Step 5: Nothing Else Broke

General knowledge probes verifying the patch didn't degrade basic capability.

In [ ]:
controls = [
    ('The capital of France is', 'Paris'),
    ('The capital of Japan is', 'Tokyo'),
    ('The chemical formula for water is H', '2'),
    ('The largest planet in our solar system is', 'Jup'),
    ('2 + 2 =', '4'),
]
print('Knowledge controls (patched model):')
all_pass = True
for prompt, expected in controls:
    ids = engine.tokenizer.encode(prompt, add_special_tokens=False)
    with torch.no_grad():
        logits = engine.forward(torch.tensor([ids], dtype=torch.long, device=engine.device))
    top_id = logits[0, -1, :].argmax().item()
    top = engine.tokenizer.decode([top_id]).strip()
    ok = top.startswith(expected) or expected.startswith(top)
    if not ok: all_pass = False
    print(f'  {"PASS" if ok else "FAIL"}: "{prompt}" -> "{top}" (expected "{expected}")')
print(f'\n{"All controls pass." if all_pass else "Some controls failed."}')

## Summary

| | |
|---|---|
| Model | Bonsai 1.7B (PrismML, Q1_0 GGUF, 240 MB) |
| Patch | 3,019 sign flips / 1.7B weights (0.00018%) |
| Patch size | ~13 KB compressed |
| Time to apply | <100 ms |
| Probes fixed | 2/2 targeted (legal clause, dosage) |
| Controls preserved | 5/5 |

The patch could be distributed as an OTA update to a million devices.
Total bandwidth: 13 MB. Equivalent LoRA adapter at 100 MB: 100 TB.